## GRPO Slides 1

- Example from Sebastion Raschka's excellent book: https://github.com/rasbt/reasoning-from-scratch

In [1]:
import sys
sys.path.append('/Users/stephen')

In [2]:
import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import load_model_and_tokenizer
from IPython.display import Latex, display
import time
import numpy as np

device = get_device()
# device = torch.device("cpu")

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

Using Apple Silicon GPU (MPS)
✓ qwen3/qwen3-0.6B-base.pth already up-to-date


In [3]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(0)
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

 47

In [4]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def reward_rlvr(answer_text, ground_truth):
    extracted = extract_final_candidate(
        answer_text, fallback=None  # Require \boxed{}
    )
    if not extracted:
        return 0.0
    correct = grade_answer(extracted, ground_truth)
    return float(correct)

In [5]:
from reasoning_from_scratch.qwen3 import KVCache
from reasoning_from_scratch.ch04 import top_p_filter


@torch.no_grad()
def sample_response(
    model,
    tokenizer,
    prompt,
    device,
    max_new_tokens=512,
    temperature=0.8,
    top_p=0.9,
):
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        device=device
        )

    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()
    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]

    generated = []
    for _ in range(max_new_tokens):
        if temperature and temperature != 1.0:
            logits = logits / temperature

        probas = torch.softmax(logits, dim=-1)
        probas = top_p_filter(probas, top_p)
        next_token = torch.multinomial(
            probas.cpu(), num_samples=1
        ).to(device)

        if (
            tokenizer.eos_token_id is not None
            and next_token.item() == tokenizer.eos_token_id
        ):
            break
        generated.append(next_token.item())
        logits = model(next_token, cache=cache)[:, -1]

    full_token_ids = torch.cat(
        [input_ids,
         torch.tensor(generated, device=device, dtype=input_ids.dtype),]
    )
    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)

In [6]:
@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):

    # Encode prompt and answer tokens separately to get the prompt length later
    prompt_ids = tokenizer.encode(prompt)
    answer_ids = tokenizer.encode(answer)
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)

    # Same as in calc_next_token_logprobas before
    logits = model(full_ids.unsqueeze(0)).squeeze(0)
    logprobs = torch.log_softmax(logits, dim=-1)

    # Index range for positions corresponding to answer tokens
    start = len(prompt_ids) - 1
    end = full_ids.shape[0] - 1

    # Same as before, except for using start and end
    t_idx = torch.arange(start, end, device=device)
    next_tokens = full_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Average over the answer token scores
    return torch.mean(next_token_logps).item()

#SW - maybe work wiht this more robust version from Raschka? Renaming here
def sequence_logprob(model, token_ids, prompt_len):
    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()
    logprobs = torch.log_softmax(logits, dim=-1)

    # Positions whose next-token probabilities we want
    # These correspond to predicting token_ids[t + 1] from position t
    start = prompt_len - 1
    end = token_ids.shape[0] - 1

    t_idx = torch.arange(start, end, device=token_ids.device)
    next_tokens = token_ids[start + 1 : end + 1]
    next_token_logps = logprobs[t_idx, next_tokens]

    # Sum log-probabilities over the answer tokens
    return torch.sum(next_token_logps), logprobs.detach(), next_tokens.detach(), next_token_logps.detach()


def compute_grpo_loss(
    model,
    tokenizer,
    example,
    device,
    num_rollouts=2,
    max_new_tokens=256,
    temperature=0.8,
    top_p=0.9,
):
    assert num_rollouts >= 2
    roll_logps, roll_rewards, samples = [], [], []
    prompt = render_prompt(example["problem"])

    was_training = model.training
    model.eval()

    for _ in range(num_rollouts):
        # Stage 1: generate rollouts
        token_ids, prompt_len, text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
        # Stage 2: compute rewards
        reward = reward_rlvr(text, example["answer"])
        
        # Stage 4: compute logprobs
        logp = sequence_logprob(model, token_ids, prompt_len)

        roll_logps.append(logp)
        roll_rewards.append(reward)
        samples.append(
            {
                "text": text,
                "reward": reward,
                "gen_len": token_ids.numel() - prompt_len,
            }
        )

    if was_training:
        model.train()

    # Stage 2: collect all rewards
    rewards = torch.tensor(roll_rewards, device=device)

    # Stage 3: compute advantages
    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

    # Stage 4: collect all logprobs
    logps = torch.stack(roll_logps)

    # Stage 5: compute policy gradient loss
    pg_loss = -(advantages.detach() * logps).mean()
    loss = pg_loss  # In the next chapter we add a KL term here

    return {
        "loss": loss.item(),
        "pg_loss": pg_loss.item(),
        "rewards": roll_rewards,
        "advantages": advantages.detach().cpu().tolist(),
        "samples": samples,
        "loss_tensor": loss,
    }


In [7]:
import json
import requests
from pathlib import Path

def load_math_train(local_path="math_train.json", save_copy=True):
    local_path = Path(local_path)
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "math_full_minus_math500/refs/heads/main/"
        "math_full_minus_math500.json"
    )

    if local_path.exists():
        with local_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        data = r.json()

        if save_copy:  # Saves a local copy
            with local_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2)

    return data

In [8]:
math_train = load_math_train()
print("Dataset size:", len(math_train))

Dataset size: 12000


In [9]:
torch.manual_seed(0)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

token_ids, prompt_len, answer_text = sample_response(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            device=device,
            max_new_tokens=512,
            temperature=0.9,
            top_p=0.9,
        )

print(answer_text)

 47


In [62]:
# #Print a bunch of problems 
# for i in range(100):
#     print(i, math_train[i % len(math_train)]['problem'], '\n')

In [11]:
# torch.manual_seed(0)

# raw_prompt = (
#     "Solve the equation $|y-6| + 2y = 9$ for $y$. "
# )
# prompt = render_prompt(raw_prompt)

# token_ids, prompt_len, answer_text = sample_response(
#             model=model,
#             tokenizer=tokenizer,
#             prompt=prompt,
#             device=device,
#             max_new_tokens=512,
#             temperature=0.9,
#             top_p=0.9,
#         )

# print(answer_text)

In [12]:
num_rollouts=8
max_new_tokens=512
temperature=0.8
top_p=0.9
lr=1e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

In [49]:
# step=2 #This is a good one
step=58 #This one is pretty good: 'Solve the equation $|y-6| + 2y = 9$ for $y$.'

optimizer.zero_grad()

current_step = step + 1
example = math_train[step % len(math_train)]

In [50]:
example

{'problem': 'Solve the equation $|y-6| + 2y = 9$ for $y$.',
 'level': 'Level 4',
 'type': 'Algebra',
 'solution': 'We consider two cases, $y\\ge 6$ and $y < 6$.\n\nCase 1: $y \\ge 6:$  If $y \\ge 6$, then $|y-6| = y-6$ and our equation is $y-6+2y=9$.  So, we have $3y = 15$, or $y=5$. However, $y=5$ does not satisfy $y\\ge 6$.  Testing $y=5$, we have $|5-6| + 2\\cdot 5 =11$, not 9, and we see that $y=5$ is not a solution.\n\nCase 2: $y < 6:$   If $y<6$, then $|y-6| = -(y-6) = -y+6$, so our equation is $-y+6+2y = 9$, from which we have $y=\\boxed{3}$. This is a valid solution, since $y=3$ satisfies the restriction $y<6$.',
 'answer': '3',
 'unique_id': 58}

In [51]:
roll_logps, roll_rewards, samples = [], [], []
prompt = render_prompt(example["problem"])

was_training = model.training
model.eval();

all_log_probs=[]
all_next_tokens=[]
all_next_token_logps=[]
for i in range(num_rollouts):
    
    torch.manual_seed(24+i) #Reproducability, 8+ is not bad, 24+ is pretty nice
    # Stage 1: generate rollouts
    token_ids, prompt_len, text = sample_response(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        device=device,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
    )
    # Stage 2: compute rewards
    reward = reward_rlvr(text, example["answer"])
    
    # Stage 4: compute logprobs
    logp, log_probs, next_tokens, next_token_logps = sequence_logprob(model, token_ids, prompt_len)
    all_log_probs.append(log_probs)
    all_next_tokens.append(next_tokens)
    all_next_token_logps.append(next_token_logps)

    roll_logps.append(logp)
    roll_rewards.append(reward)
    samples.append(
        {
            "text": text,
            "reward": reward,
            "gen_len": token_ids.numel() - prompt_len,
        }
    )

if was_training:
    model.train()

# Stage 2: collect all rewards
rewards = torch.tensor(roll_rewards, device=device)

# Stage 3: compute advantages
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

# Stage 4: collect all logprobs
logps = torch.stack(roll_logps)

# Stage 5: compute policy gradient loss
pg_loss = -(advantages.detach() * logps).mean()
loss = pg_loss  # In the next chapter we add a KL term here

In [52]:
samples

[{'text': ' $y = 1$ or $y = 2$.', 'reward': 0.0, 'gen_len': 13},
 {'text': ' To solve the equation \\( |y - 6| + 2y = 9 \\), we need to consider the definition of the absolute value function. The absolute value function \\( |y - 6| \\) can be broken into two cases based on the value of \\( y \\):\n\n1. **Case 1:** \\( y \\geq 6 \\)\n   - In this case, \\( |y - 6| = y - 6 \\).\n   - The equation becomes \\( (y - 6) + 2y = 9 \\).\n   - Simplifying, we get \\( 3y - 6 = 9 \\).\n   - Adding 6 to both sides, we get \\( 3y = 15 \\).\n   - Dividing by 3, we get \\( y = 5 \\).\n\n2. **Case 2:** \\( y < 6 \\)\n   - In this case, \\( |y - 6| = 6 - y \\).\n   - The equation becomes \\( (6 - y) + 2y = 9 \\).\n   - Simplifying, we get \\( 6 + y = 9 \\).\n   - Subtracting 6 from both sides, we get \\( y = 3 \\).\n\nWe need to check if both solutions satisfy the original equation.\n\n- For \\( y = 5 \\):\n  - \\( |5 - 6| + 2(5) = 1 + 10 = 11 \\), which does not equal 9.\n  - Therefore, \\( y = 5 \\) i

In [57]:
log_probs=all_log_probs[0]
next_tokens=all_next_tokens[0]
next_token_logps=all_next_token_logps[0]

In [58]:
log_probs.shape

torch.Size([67, 151936])

In [59]:
next_tokens

tensor([  400,    88,   284,   220,    16,     3,   476,   400,    88,   284,
          220,    17, 12947], device='mps:0')

In [75]:
for i, t in enumerate(next_tokens.tolist()):
    print(t, tokenizer.decode([t]), next_token_logps[i].item(), np.exp(next_token_logps[i].item()))
    # print((example['problem'] + tokenizer.decode(next_tokens[:i+1].tolist())).replace('$', ''))

400  $ -2.0708250999450684 0.12608170876467853
88 y -0.14767803251743317 0.8627088344229326
284  = -0.35741153359413147 0.6994845770921159
220   -0.4787667989730835 0.6195469468320092
16 1 -2.0264973640441895 0.13179634825079545
3 $ -0.4520556330680847 0.6363187683749887
476  or -0.7416161894798279 0.47634343182162747
400  $ -0.009233037941157818 0.9908094556713772
88 y -0.00861605815589428 0.9914209536981372
284  = -0.004887301009148359 0.9951246224140674
220   -0.2089933305978775 0.8114006471825641
17 2 -3.189103126525879 0.041208813461900994
12947 $. -0.8450964093208313 0.4295159468738275


In [72]:
for i, t in enumerate(next_tokens.tolist()):
    # print(t, tokenizer.decode([t]), next_token_logps[i].item(), np.exp(next_token_logps[i].item()))
    print((example['problem'] + tokenizer.decode(next_tokens[:i+1].tolist())).replace('$', ''))

Solve the equation |y-6| + 2y = 9 for y. 
Solve the equation |y-6| + 2y = 9 for y. y
Solve the equation |y-6| + 2y = 9 for y. y =
Solve the equation |y-6| + 2y = 9 for y. y = 
Solve the equation |y-6| + 2y = 9 for y. y = 1
Solve the equation |y-6| + 2y = 9 for y. y = 1
Solve the equation |y-6| + 2y = 9 for y. y = 1 or
Solve the equation |y-6| + 2y = 9 for y. y = 1 or 
Solve the equation |y-6| + 2y = 9 for y. y = 1 or y
Solve the equation |y-6| + 2y = 9 for y. y = 1 or y =
Solve the equation |y-6| + 2y = 9 for y. y = 1 or y = 
Solve the equation |y-6| + 2y = 9 for y. y = 1 or y = 2
Solve the equation |y-6| + 2y = 9 for y. y = 1 or y = 2.


In [73]:
for i, t in enumerate(next_tokens.tolist()):
    # print(t, tokenizer.decode([t]), next_token_logps[i].item(), np.exp(next_token_logps[i].item()))
    print(np.round(np.exp(next_token_logps[i].item()), 4))

0.1261
0.8627
0.6995
0.6195
0.1318
0.6363
0.4763
0.9908
0.9914
0.9951
0.8114
0.0412
0.4295


In [74]:
for i, t in enumerate(next_tokens.tolist()):
    # print(t, tokenizer.decode([t]), next_token_logps[i].item(), np.exp(next_token_logps[i].item()))
    print(tokenizer.decode([t]))

 $
y
 =
 
1
$
 or
 $
y
 =
 
2
$.


In [27]:
tokenizer.decode(next_tokens.tolist())

' The degree of the polynomial is 5.'

In [22]:
next_token_logps

tensor([-1.1131, -0.1501, -0.0054, -0.2976, -0.1021, -0.6486, -0.6494, -2.0717,
        -0.3731], device='mps:0')

In [19]:
advantages

tensor([-0.3535, -0.3535, -0.3535, -0.3535, -0.3535,  2.4742, -0.3535, -0.3535],
       device='mps:0')

In [20]:
logp

tensor(-5.4111, device='mps:0', grad_fn=<SumBackward0>)